# RecSys 2026 — Blind-A Two-Step Inference (Colab)

Runs inference on the **Blind-A** test split (80 single-turn rows) on Colab GPU, packages a CodaBench-compliant `prediction.zip` (single `prediction.json` at archive root), and saves it to Google Drive + browser download.

## Setup checklist

1. **Runtime → Change runtime type → T4 / L4 / A100 GPU**. T4 is enough for 80 rows.
2. Edit cell 4 (set `TID` to match a `config/{TID}.yaml` that points at `talkpl-ai/TalkPlayData-Challenge-Blind-A`).
3. Run all cells.

Wall time: ~2 min on A100, ~3 min on T4 (only 80 rows). Output zip is ~100 KB.

## Output layout (CodaBench-compliant)

`prediction.zip`  ← singular `prediction.json` at archive ROOT

CodaBench extracts to `/app/input/res/prediction.json`. Any other name/nesting is rejected. The script enforces this.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone from GitHub.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial

print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%nauthor:  %an%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 3) Install pinned deps.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters — edit these for a new Blind-A ship.
#
# TID         : filename (no extension) of the yaml at
#               music-crs-baselines/config/{TID}.yaml. The yaml's
#               test_dataset_name MUST be the Blind-A HF path.
# BATCH_SIZE  : 8 safe for T4, 16+ on A100. Only 80 rows so this barely
#               matters.
# ATTN        : 'sdpa' by default; 'flash_attention_2' if pip-installed.
#
# Every Blind-A ship burns one submission slot on CodaBench — see
# submission-budget rule in the plan before running.
TID = '021-two-step-wrrf-lyrics-qwen15b-blindsetA'
BATCH_SIZE = 8
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run the Blind-A two-step inference with the params from cell 4.
# Shared-encoder singleton + SDPA baked into the script; see
# colab/Run_Devset_Inference.ipynb for the full list of conventions.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_blindset.py \
    --tid {TID} \
    --eval_dataset blindset_A \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate + package into a CodaBench-compliant prediction.zip.
#
# Rule (dead-end learned previously): the zip must contain exactly
# one file at archive root named prediction.json (singular). Any other
# layout — nested dirs, plural filename — CodaBench rejects.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isfile(SRC), f'inference output missing at {SRC}'

with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (Blind-A expects 80)')
assert len(rows) == 80, f'expected 80 rows, got {len(rows)}'
sample = rows[0]
print(f'sample keys: {list(sample.keys())}')
required = {'session_id', 'user_id', 'turn_number', 'predicted_track_ids', 'predicted_response'}
missing = required - set(sample.keys())
assert not missing, f'missing keys: {missing}'
assert len(sample['predicted_track_ids']) == 20, f'expected 20 tids, got {len(sample["predicted_track_ids"])}'
assert sample['predicted_response'].strip(), 'empty predicted_response'

# Stage under a clean directory so the zip has a flat root.
stage = '/content/_stage_prediction'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, 'prediction.json'))  # singular, at root

# Explicit zip (not shutil.make_archive — we want a flat layout).
!cd {stage} && rm -f /content/prediction.zip && zip -q /content/prediction.zip prediction.json
!unzip -l /content/prediction.zip
print('\nprediction.zip ready — CodaBench-compliant.')

In [ ]:
# 7a) Download prediction.zip via browser popup.
from google.colab import files
files.download('/content/prediction.zip')

In [ ]:
# 7b) Mount Drive + save both the ship zip and the raw JSON.
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
dst_dir = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst_dir, exist_ok=True)

# Tag the Drive-copied zip with TID so multiple Blind-A ships don't clobber.
shutil.copy('/content/prediction.zip', f'{dst_dir}/{TID}__prediction.zip')
shutil.copy(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json', dst_dir)

print(f'saved to Drive:')
!ls -lh {dst_dir}

## Done. To ship to CodaBench:

1. Upload `prediction.zip` (from 7a or Drive) to the CodaBench submission page.
2. Wait for scoring — typically 5–10 min.
3. On score return, log the row to `documents/submissions_log.md` with `tag=[blindA]`.
4. Archive the (query, predicted_response, score) triples into `documents/blind_responses_scored.md` per plan §2.5.4 (Judge-behavior mining).

## Local pickup of raw JSON (optional, for post-hoc analysis):

```bash
cd /Users/orrimoch/PythonProjs/recsys2026
TID=021-two-step-wrrf-lyrics-qwen15b-blindsetA
cp ~/Google\ Drive/My\ Drive/recsys2026-predictions/${TID}.json \
   music-crs-baselines/exp/inference/blindset_A/
```

## Reminder: submission budget

Per plan §2.6, total Blind-A submissions are capped. Confirm headroom before shipping; `scripts/validate_prediction.py` has the cap check.